In [32]:
!pip install -q groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 5.7 MB/s eta 0:00:00


In [43]:
import json
import requests
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

# Dicionário de Gêneros principal
GENRES = {16: "Animação", 35: "Comédia", 10751: "Família", 14: "Fantasia", 18: "Drama", 10749: "Romance", 99: "Documentário", 12: "Aventura", 28: "Ação"}

def obter_filmes_da_ia(prompt_usuario, historico_anterior=""):
    """Agora a IA recebe o histórico para não repetir filmes e refinar a busca."""
    prompt_sistema = (
        "Você é um curador de cinema. O usuário quer filmes baseados no humor. "
        "REGRAS: 1. Sugira apenas filmes REAIS. 2. Se for refinamento, mude o estilo conforme o pedido. "
        "3. Não repita filmes já sugeridos no histórico. "
        "Retorne um JSON: {'mensagem': 'texto curto', 'lista_filmes': ['Filme 1', 'Filme 2', 'Filme 3']}"
    )

    conteudo_usuario = f"Histórico: {historico_anterior}\nPedido atual: {prompt_usuario}"

    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": conteudo_usuario}
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.5,
        response_format={"type": "json_object"}
    )
    return json.loads(chat_completion.choices[0].message.content)

def buscar_streaming(movie_id):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers?api_key={TMDB_API_KEY}"
    try:
        res = requests.get(url).json()
        br = res.get('results', {}).get('BR', {}).get('flatrate', [])
        return [p['provider_name'] for p in br] if br else ["Aluguel/Indisponível"]
    except: return ["Erro ao buscar"]

def buscar_detalhes_tmdb(nome_filme):
    url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={nome_filme}&language=pt-BR"
    res = requests.get(url).json()
    return res['results'][0] if res.get('results') else None

# --- CHAT INTERATIVO ---

def chatbot_interativo():
    print("🤖 Bem-vindo ao seu cinema particular! (Digite 'sair' para encerrar)")
    historico = []

    while True:
        user_input = input("\nVocê: ")

        if user_input.lower() in ['sair', 'exit', 'quit']:
            print("🤖 Espero que você aproveite o filme! Até a próxima. ✨")
            break

        try:
            print("🎬 Analisando opções...")
            # Passamos o histórico de filmes já sugeridos para a IA
            sugestao = obter_filmes_da_ia(user_input, ", ".join(historico))
            print(f"\n✨ {sugestao['mensagem']}")

            for nome in sugestao['lista_filmes']:
                historico.append(nome) # Adiciona ao histórico para evitar repetição
                f = buscar_detalhes_tmdb(nome)
                if f:
                    streamings = buscar_streaming(f['id'])
                    generos = [GENRES.get(g, "Geral") for g in f.get('genre_ids', [])[:2]]

                    print(f"\n🎥 {f['title']} ({f.get('release_date', '????')[:4]})")
                    print(f" {', '.join(generos)} | ⭐ {f['vote_average']}")
                    print(f" Onde: {', '.join(streamings)}")
                    print(f" Por que esta é uma boa escolha: {f.get('overview', '...')[:140]}...")

            print("\n" + "-"*30)
            print("👉 Gostou desses? Se quiser algo diferente (ex: 'quero animações' ou 'mais antigos'), é só pedir!")

        except Exception as e:
            print(f"[Erro]: {e}")

chatbot_interativo()

🤖 Bem-vindo ao seu cinema particular! (Digite 'sair' para encerrar)

Você: estou triste, quero um filme para me animar
🎬 Analisando opções...

✨ Espero que esses filmes te animem!

🎥 O Máskara (1994)
 Romance, Comédia | ⭐ 6.977
 Onde: Netflix, HBO Max, Netflix Standard with Ads, HBO Max Amazon Channel, Telecine Amazon Channel
 Por que esta é uma boa escolha: Em Edge City vive Stanley Ipkiss, um cara decente que trabalha em um banco, mas é socialmente desajeitado e sem muito sucesso com as mulhere...

🎥 Debi & Lóide: Dois Idiotas em Apuros (1994)
 Comédia | ⭐ 6.645
 Onde: Amazon Prime Video, Telecine Amazon Channel, Amazon Prime Video with Ads
 Por que esta é uma boa escolha: Dois amigos debilóides vão para Aspen, no estado do Colorado, para tentar devolver uma maleta esquecida pela passageira da limusine que um d...

🎥 O Grande Hotel Budapeste (2014)
 Comédia, Drama | ⭐ 8.0
 Onde: Disney Plus, Netflix, Netflix Standard with Ads
 Por que esta é uma boa escolha: No período entre as duas 